# Capstone — mirrors the deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Title —** Which Content Pages Should Editors Review First?
Comparing a hand-written prioritisation rule with machine-learning models on client-held-out data.

**Live paper URL** — https://flyrank-ml.vercel.app/

**Repository —** https://github.com/Lakes41/flyrank-ml

**Author —** Amir Oyeleke, Machine Learning Engineer. GitHub: https://github.com/lakes41/ · LinkedIn: https://www.linkedin.com/in/amir-oyeleke-5b3168296 · Email: Oyelekeamir123@gmail.com

**Data credit —** Built on the FlyRank ML Internship programme dataset (FlyRank: https://flyrank.ai).

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.
> Working with the starter export only? Every result below is reproducible from the `work/outputs/` JSONs the weekly notebooks produced — no warehouse access required.


## 1. Question

**The research question, in plain English.**

> Every month an editor has time to review perhaps 200 content pages. Given thousands of candidate pages from multiple clients, which ones should they review first so that the limited review time goes to pages showing a genuine severe traffic decline?

**The decision this work supports.** The output is a ranked queue. The top pages go to a REFRESH review; the next 1,500 are flagged for OBSERVE monitoring; the rest are IGNORED for this month's cycle. The rank is decision-support only — no row is auto-archived or auto-republished.

**Stripped to its core, we are evaluating:**
- a frozen **hand-written baseline rule** (developed in Week 4) against
- a **Logistic Regression** and
- a **Random Forest** (200 trees, max_depth=6, balanced subsample weighting, 11 features)

using **client-held-out validation** (entire clients excluded from training so the model cannot look good by memorising a client's typical traffic pattern).


In [ ]:
# --- Capstone: utility setup (robust to where you launch the notebook) ---
# This notebook re-reads the weekly output JSONs and reconstructs every claim the deployed
# paper makes. Run Runtime → Run all before editing.
import json, pathlib, textwrap, os, sys
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

# ----------------------------------------------------------------
# Robust repository-root detection.
# Walk up from the notebook file's real location (not from os.getcwd()).
# Jupyter guarantees __file__ is NOT set inside a notebook, so we use:
#   1) if IPython is available, resolve from the IPython session's starting dir
#   2) else try a list of likely roots (handles both Colab and local launch)
# ----------------------------------------------------------------
def _find_repo_root(marker_rel="work/outputs/model_comparison.json", max_steps=6):
    """Return the repository root directory, or raise FileNotFoundError.

    Strategy:
      A. Try the notebook's directory as reported by IPython (works in real Jupyter).
      B. Walk up from os.getcwd() searching for marker_rel.
      C. Walk up from this process's executable directory as a last resort.
    """
    candidates = []
    # A. IPython.utils.path.locate_profile / pwd
    try:
        import IPython
        ipy = IPython.get_ipython()
        if ipy is not None and hasattr(ipy, "starting_dir"):
            candidates.append(pathlib.Path(ipy.starting_dir).resolve())
        # Use IPython's current cwd if known
        candidates.append(pathlib.Path.cwd().resolve())
    except Exception:
        candidates.append(pathlib.Path.cwd().resolve())
    # B. Walk-up from cwd
    probe = pathlib.Path.cwd().resolve()
    for _ in range(max_steps):
        candidates.append(probe)
        probe = probe.parent
    # C. Process dir
    try:
        candidates.append(pathlib.Path(sys.argv[0]).resolve().parent)
    except Exception:
        pass
    seen = set()
    for c in candidates:
        key = str(c)
        if key in seen: continue
        seen.add(key)
        p = c / marker_rel
        if p.exists(): return c
        # if user cloned into flyrank-ml subdir, c/../ might be parent-of-repo; check nested
        p2 = c / "flyrank-ml" / marker_rel
        if p2.exists(): return (c / "flyrank-ml").resolve()
    # Colab fallback path if user clones per instructions below
    for colab in ["/tmp/fly", "/content/flyrank-ml", "/content"]:
        if (pathlib.Path(colab) / marker_rel).exists():
            return pathlib.Path(colab)
    raise FileNotFoundError(
        "Could not locate the FlyRank repository root. Please ensure work/outputs/model_comparison.json\n"
        "is reachable from the current working directory, or on Colab run:\n"
        "   ! git clone --depth 1 https://github.com/Lakes41/flyrank-ml.git /tmp/fly"
    )

ROOT = _find_repo_root().resolve()
OUT  = ROOT / "work" / "outputs"

# If working in Colab: clone the repo instead and this section will self-detect.
# ! git clone --depth 1 https://github.com/Lakes41/flyrank-ml.git /tmp/fly

def load(name):
    p = OUT / name
    if not p.exists():
        raise FileNotFoundError(
            f"Missing work/outputs/{name}. Expected at: {p}\n"
            f"Make sure the five weekly-receipt JSONs exist in work/outputs/ before running this cell."
        )
    return json.loads(p.read_text())

mc  = load("model_comparison.json")     # 5-fold fair comparison
bm  = load("baseline_metrics.json")     # baseline rule + signal audits
va  = load("validation_audit.json")     # split regimes + leakage audit + claim rewrites
ps  = load("playbook_summary.json")     # playbook action distribution + archetypes
sa  = load("signal_audit_receipt.json") # slice facts + staleness / pos×SV verdicts


# String-constant aliases so the escaping below is readable and never mangled by JSON or template layers.
_BSLASH = chr(92)   # backslash
_PIPE   = chr(124)  # vertical pipe
_NL     = chr(10)   # newline
_CR     = chr(13)   # carriage return
BSLASH, PIPE, NL, CR = _BSLASH, _PIPE, _NL, _CR
def _md_escape(v):
    """Minimal Markdown table escaping — pipes, newlines, backslashes, leading/trailing whitespace."""
    if v is None:
        return ""
    s = str(v)
    s = s.replace(BSLASH, BSLASH + BSLASH)
    s = s.replace(PIPE, BSLASH + PIPE)
    s = s.replace(NL, "<br>")
    s = s.replace(CR, "")
    return s.strip()

def _md_table(rows, columns):
    """Build a GitHub-flavored Markdown table. Zero external dependencies — works in any IPython frontend."""
    cols = [_md_escape(c) for c in columns]
    head = PIPE + " " + (" " + PIPE + " ").join(cols) + " " + PIPE
    sep  = PIPE + " " + (" " + PIPE + " ").join("---" for _ in cols) + " " + PIPE
    body = []
    for r in rows:
        cells = [_md_escape(v) for v in r]
        while len(cells) < len(cols):
            cells.append("")
        cells = cells[:len(cols)]
        body.append(PIPE + " " + (" " + PIPE + " ").join(cells) + " " + PIPE)
    return NL.join([head, sep] + body)

def show_table(title, rows, columns):
    """Render a table using plain Markdown. No pandas Styler, no jinja2, no extra installs needed."""
    display(Markdown(f"**{title}**"))
    display(Markdown(_md_table(rows, columns)))
    return None

LANE_ROWS    = int(sa["setup"]["lane_rows"])
STARTER_ROWS = int(sa["setup"]["starter_rows"])
LANE_CLIENTS = int(sa["setup"]["lane_clients"])
BASE_RATE    = round(100 * float(sa["setup"]["lane_base_rate_severe_decline"]), 1)
print(f"Repository ROOT resolved to : {ROOT}")
print(f"Starter-export rows shipped  : {STARTER_ROWS:,}")
print(f"Lane-2 eligible rows          : {LANE_ROWS:,}  ({LANE_ROWS/STARTER_ROWS*100:.1f}% of starter)")
print(f"Distinct Lane-2 clients      : {LANE_CLIENTS}")
print(f"Label base rate (severe_decline): {BASE_RATE}%")

# --- Remember where this utility cell came from, so later cells can re-run it if needed.
import json as _json, pathlib as _pathlib
try:
    import IPython as _IPY
    _ipy = _IPY.get_ipython()
    if _ipy is not None and hasattr(_ipy, "user_ns"):
        # Save the source so cell-04 can re-execute it if globals are missing.
        # (Reconstructed from this notebook json on disk using the json file at ROOT.)
        _nb_path = _pathlib.Path(ROOT) / "work" / "notebooks" / "capstone.ipynb"
        if _nb_path.exists():
            _nb = _json.loads(_nb_path.read_text())
            _cells = _nb.get("cells", [])
            if len(_cells) >= 3:
                _c2 = _cells[2]
                _src2 = _c2["source"]
                if isinstance(_src2, list):
                    _src2 = "".join(_src2)
                _ipy.user_ns["_capstone_cell_02_source"] = _src2
                print(f"[capstone cached utility-cell source ({len(_src2)} chars) for out-of-order recovery]")
except Exception:
    pass


## 2. Data

**Which release, which tables, date windows, what we excluded and why — public-safe only, no client names or raw queries.**

**Unit of observation (one row =):** a single content page, identified by an anonymised `content_id`, from a 90-day Search Console aggregate snapshot.

**Release used:** the anonymised **FlyRank ML Internship starter dataset** (hereafter "starter export" — a single anonymised CSV with no client names, no URLs, no raw queries). 30,000 rows shipped across 32 clients.

**Why three eligibility filters were applied before any training or audit:**
1. **Minimum traffic requirement.** Pages with fewer than 100 impressions had too little activity to rank reliably.
   Rule: `impressions_90d >= 100`
2. **Unreliable placeholder ranks on low-traffic rows.** Search Console writes position = 0 when the rank is unknown or unreliable — position `0` is a *placeholder* indicating missing ranking information, not actual search position #1. We excluded rows where `avg_position = 0 AND impressions_90d < 500`.
3. **Enough age for a decline trend to be meaningful.** Pages younger than 90 days cannot show a forward-30-day severe decline with enough history. Rule: `content_age_days >= 90`.

**After filters:** 22,006 pages from 30 clients remained (73.3% of the starter export). Base rate of severe_decline on this slice = **59.7%**.

**Past information → decision point → future outcome timeline.**
- **Features (before):** trailing 90d Search Console signals (impressions, clicks, CTR, average search position) + article metadata (age, days since update, word-count availability, word count when known) + keyword search-volume estimates.
- **Decision point:** editors rank pages and pick 200 to review. No forward-month information is available yet; this is the exact moment the model must operate at in real use.
- **Label (after):** forward-30-day trend indicator. `severe_decline = 1` when `trend_pct < -20%`.

**Feature/label window guardrails.** Feature windows and label windows are drawn from disjoint calendar ranges. Every feature column is computed from the 90-day aggregate on the left side of the decision point; every label column from the 30-day window on the right side. This is the primary guard against leakage.


In [ ]:
# --- 2. Data facts rebuilt from JSON receipts ---
# Safety: if the user re-runs this cell out of order, re-check required globals exist, else rebuild.
from __main__ import __dict__ as _G
_REQUIRED = ["show_table","LANE_ROWS","STARTER_ROWS","LANE_CLIENTS","BASE_RATE","sa","ROOT"]
if any(k not in _G or _G[k] is None for k in _REQUIRED):
    print("[capstone] required globals missing; re-running utility cell … please run cell [02] manually if this recurs.")
    # Primary path: re-execute cached utility-cell source directly into the user's namespace.
    _utility_src = None
    try:
        import IPython; _ipy = IPython.get_ipython()
        if _ipy is not None and hasattr(_ipy, "user_ns"):
            _utility_src = _ipy.user_ns.get("_capstone_cell_02_source")
            if _utility_src:
                # Run synchronously in _G
                exec(compile(_utility_src, "<capstone-cell-02-replay>", "exec"), _G)
    except Exception as _e:
        print(f"   (IPython path failed: {type(_e).__name__}: {_e})")
    if not _utility_src:
        # Fallback path: reconstruct utility source from the notebook JSON on disk.
        try:
            import json as _J, pathlib as _P
            _probe_paths = []
            if "ROOT" in _G and _G["ROOT"]:
                _probe_paths.append(_P.Path(_G["ROOT"]) / "work/notebooks/capstone.ipynb")
            _probe_paths += [_P.Path.cwd() / "capstone.ipynb", _P.Path.cwd() / "work/notebooks/capstone.ipynb"]
            for _pp in _probe_paths:
                if _pp.exists():
                    _nb = _J.loads(_pp.read_text())
                    _cells = _nb.get("cells", [])
                    if len(_cells) >= 3:
                        _s = _cells[2]["source"]
                        _utility_src = "".join(_s) if isinstance(_s, list) else _s
                        break
            if _utility_src:
                exec(compile(_utility_src, "<capstone-cell-02-replay-disk>", "exec"), _G)
                print("   (re-ran utility cell from disk-based notebook source)")
            else:
                print("   (ERROR: could not re-run utility cell. Please run cell index 02 first.)")
        except Exception as _e2:
            print(f"   (disk fallback failed: {type(_e2).__name__}: {_e2})")
# end of out-of-order recovery guard.

rows = [
    ["Starter export rows shipped",          f"{STARTER_ROWS:,}",             "32 clients"],
    ["Lane-2 eligible rows (after filters)", f"{LANE_ROWS:,}",                f"{LANE_CLIENTS} clients"],
    ["Lane-2 slice share of starter",        f"{LANE_ROWS/STARTER_ROWS*100:.1f}%",  "73.3% of 30k starter rows"],
    ["Label base rate (severe_decline)",     f"{BASE_RATE}%",                 "trend_pct < -20%"],
]
show_table("Slice eligibility and counts",
           rows, ["Fact", "Value", "Notes"])

show_table("Three Lane-2 eligibility filters (public-safe)", [
    ["Minimum traffic",
     "Pages with fewer than 100 impressions are too small to rank reliably.",
     "impressions_90d >= 100"],
    ["No unreliable placeholder ranks on tiny rows",
     "position=0 is a placeholder for missing/unreliable rank information, not actual rank #1.",
     "NOT (avg_position = 0 AND impressions_90d < 500)"],
    ["Enough age to observe a trend",
     "Forward-30-day severe-decline labels need prior history ≥90d for staleness semantics to be meaningful.",
     "content_age_days >= 90"],
], ["Filter name", "Why it was applied", "Technical rule"])

show_table("Excluded feature families (what was removed, and why)", [
    ["Forward trend outcomes",
     "trend_pct, trend_direction, and any decline-named siblings",
     "They directly describe the future outcome the model is supposed to predict."],
    ["Label siblings and decline-flag columns",
     "Any column with 'decline' in the name",
     "Effectively the label under a different alias."],
    ["30-vs-prior-30 ratio columns",
     "Any 30-over-prior-30 ratio derived inside the label window",
     "Overlaps with how trend_pct is constructed — leaks part of the answer."],
    ["Editorial/product decision flags shipped outside panel",
     "Any editorial-flag / product-score column originating outside training",
     "May already encode future-trend knowledge from a human that was not a fair input."],
    ["Domain/URL/raw query/PII",
     "client_domain, raw query, any PII (none shipped in starter export)",
     "Privacy and reproducibility — identity must never anchor predictions."],
    ["Row counts / revenue / figures computed inside label month",
     "Any in-label-month aggregations",
     "Violate before/after timeline (numbers come from the label period)."],
    ["Label-month 'has_position=1' style coverage flags",
     "Coverage flags computed during label month (currently unverifiable on starter CSV)",
     "Would describe coverage during the label window, potentially leaking forward data."],
    ["Columns derived from the same last-30-day totals used to build trend_pct",
     "Any last-30-days totals shared with trend_pct construction",
     "Encode pieces of the label's numerator or denominator — partial access to the answer."],
], ["Family name", "Technical fields", "Reason for exclusion"])


## 3. Methodology

Seven natural questions, in order.

### 3.1 What is the model trying to identify?

The **label** is the outcome the model is trying to identify. In this work the label is `severe_decline`, a boolean that turns `1` when the 30-day forward trend is worse than -20%:

```
severe_decline = 1  if  trend_pct < -20%
```

Interpreted: "this page's trajectory fell steeply enough in the forward month that an editor should consider reviewing it early."

### 3.2 What information does the model use?

**Features** are the information available to the model when it ranks a page. All are measured before the decision point; nothing from the future is allowed in. Eleven columns enter every model:

1. log-scaled impressions (`log_impressions_90d`)
2. click-through rate (`ctr_filled`)
3. average search position (`position_filled` — with missing-observation fill-in)
4. log-scaled keyword search volume (`log_search_volume`)
5. content age in days (`age_days`)
6. days since the last article update (`days_since_update`)
7. staleness bucket (`staleness_bucket`: 0/1/2 for <90d / 90–179d / ≥180d)
8. visibility bucket (`vis_bucket`: 0/1/2 from traffic tier)
9. striking-distance bonus (`striking_bonus`: 0/1 flag for page on cusp of a top-rank jump)
10. word-count availability flag (`has_word_count`)
11. median-imputed word count when known (`word_count_filled`)

### 3.3 What was the existing approach?

The **baseline** is the existing hand-written baseline developed in Week 4 (it is the comparison point — a new model must beat this baseline by a clear margin before it can replace the rule). The score combines three integer components:

```
baseline_score = staleness_bucket(0..2) + vis_bucket(0..2) + striking_bonus(0/1)
```
and turns it into REFRESH / OBSERVE / IGNORE using thresholds ≥3 / =2 / <2.

### 3.4 What models were tested?

Two statistical models on top of the same 11 features on the same 5 folds:
- **Logistic Regression** with standardised inputs (a sensible interpretable first model).
- **Random Forest Classifier:** 200 trees, `max_depth=6`, `class_weight='balanced_subsample'`, `min_samples_leaf=5`, `random_state=42`. A modest-capacity forest deliberately kept small to avoid overfit on a client-structured slice.

### 3.5 How was the test kept fair?

**Client-held-out validation** means entire clients are excluded from training and used only for testing. The model cannot memorise a client's typical traffic baselines and still look good on held-out pages from that same client. Technical configuration:

```python
GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
# split key = client_id  (not page_id)
```

Five folds, 80% train / 20% test *by client*.

### 3.6 How was cheating / leakage prevented?

**Leakage** is information from the outcome or the future accidentally entering the model inputs and making performance look better than it really is. A 3-taxonomy, 9-point leakage audit was run:

| Taxon | Attack |
|---|---|
| 1. Label-derived columns | Any column built from the trend_pct numerator/denominator |
| 2. Overlapping windows | Feature window creeping into the label calendar window |
| 3. Population-selection bias | Label distribution biased by rows that only exist when the decline indicator fires |

Harness confession jump: when `trend_pct_forward` was deliberately put back, precision@200 jumped +32.5 percentage points. That is the magnitude of leakage we are protecting against.

### 3.7 How was performance measured?

If editors can review only 200 pages, **precision@200** measures how many of those 200 recommendations are genuinely severe-decline pages. Formally:

```
precision@K = (# of true severe_decline rows in the top-K ranked positions) / K
```

Primary metric: precision@200. Secondary metric: precision@50 (quality of the very top of the queue, where REFRESH cuts are drawn).


In [ ]:
# --- 3. Methodology rebuilt from outputs ---
display(Markdown("### Feature columns entering every model"))
print(", ".join(mc["feature_columns"]))

display(Markdown("### Client-held-out GroupShuffleSplit config"))
print(f"  {mc['split']}  ·  random_state = {mc['random_state']}")

display(Markdown("### Baseline rule (developed in Week 4)"))
print(f"  Score components  : {bm['baseline_rule']['score_components']}")
print(f"  REFRESH threshold : score >= {bm['baseline_rule']['action_thresholds']['REFRESH'].strip('score>=') or 3}  → top {ps['action_cuts']['REFRESH_top_K']} rows/month")
print(f"  OBSERVE threshold : score = 2                                              → next {ps['action_cuts']['OBSERVE_next_K']:,} rows")
print(f"  IGNORE  threshold : score < 2                                              → rest")

display(Markdown("### Leakage harness confession jump (taxon-1 sanity-check)"))
l_audit = va["leakage_audit"]["taxon_1_label_derived"]
rows = [
    ["Honest (label-derived features excluded)", f"{100*l_audit['honest_prec200']:.1f}%"],
    ["With trend_pct_forward deliberately added back (confession)", f"{100*l_audit['leaked_prec200']:.1f}%"],
    ["Confession jump (what we are protecting against)", f"+{l_audit['harness_confession_jump_percentage_points']:.1f} percentage points"],
    ["Intersection of suspect-set with honest feature-set empty?", "✓ YES" if l_audit["feature_suspect_intersection_empty"] else "✗ NO — LEAK RISK"],
]
show_table("Leakage audit (9-point harness; taxon 1 of 3 shown as illustration)",
           rows, ["Condition", "precision@200 on the harness slice"])

# Reproduce 5-fold honest split summary
display(Markdown("### 5-fold honest split sizes (public-safe counts)"))
rows_m = []
for fold in range(5):
    pf = [p for p in mc["per_fold_results"] if p["fold"] == fold and p["method"].startswith("Baseline")][:1]
    if pf:
        rows_m.append([fold, pf[0]["n_test"]])
show_table("Per-fold held-out client-slice sizes",
           rows_m, ["Fold idx", "Test rows (pages held out)"])


## 4. Results (vs baseline)

**Headline takeaway in one sentence.** Within a 200-page monthly review capacity, the **Random Forest** identified about **151 severe-decline pages**, compared with about **106** using the hand-written baseline: a gain of ~45 genuinely-declining pages per cycle.

Technical form on the 5 client-held-out folds:
- Random Forest (200 trees, max_depth=6) → **precision@200 = 75.6% ± 8.4%**
- Logistic Regression (scaled) → **precision@200 = 73.5% ± 6.9%**
- Hand-written baseline (Week-4 frozen rule) → **precision@200 = 52.9% ± 15.5%**
- Base rate (label by chance) → **59.7%**

RF beats baseline by +22.7 percentage points; LogReg is within one standard deviation of RF (+2.1 pp gap), so either model is defensible operationally.

**Why the split design matters (14.2 pp inflation on a naive shuffle).**
A simple `ShuffleSplit` (pages randomly assigned to train/test without respecting client boundaries) inflates the Random Forest precision@200 to **89.8% ± 1.6%** because the model can memorise client-level traffic means and still look good on held-out individual pages from those clients. The honest GroupShuffleSplit result is the lower 75.6% number. That 14.2 pp gap is the exact reason the paper insists on grouped validation.


In [ ]:
# --- 4. Main result tables rebuilt from JSON ---
display(Markdown("### 4.1 Honest comparison table (5 client-held-out folds, same 11 features)"))
comp_rows = [[r["method"], r["prec@200 (mean±std)"], r["prec@50 (mean±std)"], r["mean_n_test"]] for r in mc["comparison_table"]]
show_table("Model vs baseline on the same folded GroupShuffleSplit (primary metric = prec@200)",
           comp_rows, ["Method", "prec@200 (mean ± std)", "prec@50 (mean ± std)", "Mean test rows per fold"])

display(Markdown("### 4.2 Split-design comparison (same Random Forest, two split regimes)"))
reg_rows = [[r["regime"], r["prec@200 (mean±std)"], r["prec@50 (mean±std)"], r["mean_test_base_rate"]] for r in va["comparison_table"]]
show_table("Naive random split vs honest client-held-out GroupShuffleSplit",
           reg_rows, ["Split regime", "prec@200 (mean ± std)", "prec@50 (mean ± std)", "Mean test-set base rate"])

display(Markdown("### 4.3 Interpretation numbers"))
# Rebuild headline counts from the rates
topK = 200
rf_200 = 0.756
base_200 = BASE_RATE / 100
rule_200 = 0.529
rows = [
    ["Random Forest (RF)    severe-decline pages in top 200", round(topK*rf_200),   f"{100*rf_200:.1f}% prec@200"],
    ["Hand-written baseline severe-decline pages in top 200", round(topK*rule_200), f"{100*rule_200:.1f}% prec@200"],
    ["Label-by-chance severe-decline pages in top 200",       round(topK*base_200), f"{100*base_200:.1f}% base rate"],
    ["RF vs baseline (additional useful candidates)",         round(topK*(rf_200-rule_200)), "≈45 pages/month saved"],
    ["RF vs label-by-chance (net value beyond random)",       round(topK*(rf_200-base_200)), f"+{round(100*(rf_200-base_200),1)} pp over the base rate"],
    ["RF over LogReg gap (suggestive, within one std)",       round(topK*(0.756-0.735)),  "+2.1 pp = ~4 extra pages at top 200"],
]
show_table("Interpreting the 5-fold means as 'pages I actually get to review'",
           rows, ["Fact", "Count at top 200", "Rate / notes"])


## 5. Limitations

**Honest scope:** this work is evidence-based, associative, decision-support, and deliberately narrow. It does not claim traffic recovery, causality, or guaranteed future performance on a warehouse.

1. **Starter-export scope only.** Results were verified against the public-safe aggregated 90-day starter snapshot. Forward-rolling time-based warehouse validation was not run; none of the claims extend beyond what this snapshot can support.
2. **Label construction is proxy-level.** `severe_decline = trend_pct < -20%` is a practical proxy, not a business-truth label. A 20% drop on a 10-impression page is not the same severity as a 20% drop on a 100k-impression page; the bucket rule treats them the same.
3. **Staleness ≥ 180d cell is very small (n = 35, 1.6‰ of the slice).** The monotonicity signal is real, but any recommendation based on the ≥180d bucket alone is a small-cell extrapolation.
4. **Grouped split (by client) but not time-based.** We held out *clients*, not *later calendar months*. On a warehouse, the honest production-style validation is a forward-calendar holdout; that more honest check was not available on the starter export.
5. **Probabilities are ranked, not calibrated.** We use the order of RF scores to draw the 500 / 1,500 / 20,006 cuts, not their absolute `p_severe_decline` value. No Platt scaling or isotonic calibration was applied.
6. **Content-depth signal is directionally reversed in this slice.** High word count co-occurs with severe decline on the starter slice, so the model weights it as a decay correlate rather than a quality signal. This is a slice-specific caveat; future content-type flags may undo it.
7. **Heavy-tail dominance.** Top-1% of pages by impressions capture ~21% of 90-day traffic by share. The top 1% by clicks hold ~30% of click share. Generalisations to the tail must be stated carefully.
8. **Missing feature: explicit keyword intent.** No intent-class or SERP-feature column was shipped in the starter export. Comparing "informational articles only" vs "commercial pages" would materially sharpen both RF and LogReg precision; it is currently absent from every model here.

Anything outside these eight boundaries is outside the claim set of the paper.


In [ ]:
# --- 5. Limitations quantified from receipts ---
display(Markdown("### 5.3 Staleness ≥ 180d cell size (limitation quantification)"))
sig1 = bm["signal_1_staleness"]
rows = []
for b in sig1["buckets"]:
    rows.append([b["staleness_bucket"], f"{b['n']:,}", f"{100*b['n']/LANE_ROWS:.2f}% of slice",
                 f"{b['severe_decline_rate']}%", f"{b['median_trend_pct']}%"])
show_table("Staleness buckets, sizes, and severe-decline monotonic rise",
           rows, ["Staleness bucket", "Rows (n)", "Slice share", "Severe-decline rate", "Median trend_pct"])

display(Markdown("### 5.7 Heavy-tailed traffic shares (top-1% of pages)"))
s1 = sa["section_1_distributions"]
rows = [
    ["Impressions share, top-1% pages",  f"{s1['impressions_top1pct_share']:.2f}%"],
    ["Clicks share,      top-1% pages",  f"{s1['clicks_top1pct_share']:.2f}%"],
    ["Word-count share,  top-1% pages",  f"{s1['word_count_top1pct_share']:.2f}%"],
]
show_table("Heavy-tail concentration (limitation 7 quantified)", rows, ["Fact", "Share"])

# Human-readable summary of all eight limitations for the paper
display(Markdown("""
### Full limitations list (verbatim from the paper)
1. **Starter-export scope only.**
2. **Label construction is proxy-level.**
3. **Staleness ≥ 180d cell is very small (n = 35, 1.6‰ of the slice).**
4. **Grouped split (by client) but not time-based.**
5. **Probabilities are ranked, not calibrated.**
6. **Content-depth signal is directionally reversed in this slice.**
7. **Heavy-tail dominance.**
8. **Missing feature: explicit keyword intent.**
"""))


## 6. Ranked recommendations

The output is a practical **content-action playbook**, not a production pipeline. Every row gets one of three actions, with deliberate guardrails on what must stay human-reviewed.

### Population distribution across 22,006 rows
| Action | Rows | Slice share | Interpretation |
|---|---|---|---|
| REFRESH | 500 | 2.3% | Send to editorial review now. The top of the ranked queue. |
| OBSERVE | 1,500 | 6.8% | Watch this month; promote to REFRESH next cycle if staleness or CTR delta worsens. |
| IGNORE | 20,006 | 90.9% | Leave alone for this cycle; no action required. |

### 5 ranked recommendations (paper's §6 Recommendations section)
1. **Prioritise the top-500 REFRESH queue by archetype tier. VERY_STALE_HIGH_VIS first (n = 12; small, high value).** Then STRIKING_DISTANCE (n = 282) and CONTENT_DEPTH_GAP (n = 306), then LOW_CTR_DECAY (n = 13,027) and MIXED_SIGNALS (n = 8,379).
2. **Retire the hand-written baseline's pure-integer 0..5 ranking from operational queues.** Keep the rule's three components — staleness bucket, visibility bucket, striking-distance bonus — as model features and as an auditable fallback when the model score is unavailable.
3. **For mid-queue (OBSERVE, ranks 501–2,000): apply a CTR-vs-position-tier-delta check first before promoting to REFRESH.** A page that is still in a high-rank position with a CTR below its tier median is a better REFRESH candidate than a page that is low-rank *and* low-CTR (the latter is naturally decaying and less salvageable).
4. **Add a content-type flag in the model and drop raw age_days from a future release for news / feedly items.** Long-form cornerstone content benefits from age-related staleness semantics that news items legitimately do not.
5. **Do not deploy the score into any automated action without first passing a forward-month time-based holdout on the warehouse.** Per Limitation 4, the current GroupShuffleSplit-validated score is decision-support only.

### Explicit no-go list — never hand off to an automaton
- ❌ auto-delete or auto-archive pages
- ❌ auto-publish updates to a CMS
- ❌ auto-rewrite SEO keywords, headlines, or titles
- ❌ billing or pricing decisions of any kind
- ❌ strategic pages (defined by the client) without human sign-off before action

These are *decision support outputs only*. Every action described above is taken by an editor, not by a script.


In [ ]:
# --- 6. Playbook rebuilt from playbook_summary.json ---
display(Markdown("### 6.1 Action distribution on the 22,006-row slice"))
acts = ps["actions"]
action_rows = [
    ["REFRESH", acts["REFRESH"],  f"{100*acts['REFRESH']/LANE_ROWS:.1f}%",  f"Top {ps['action_cuts']['REFRESH_top_K']:,} rows by model rank; go to editorial review"],
    ["OBSERVE", acts["OBSERVE"],  f"{100*acts['OBSERVE']/LANE_ROWS:.1f}%",  f"Next {ps['action_cuts']['OBSERVE_next_K']:,} rows by model rank; monitor-cycle only"],
    ["IGNORE",  acts["IGNORE"],   f"{100*acts['IGNORE']/LANE_ROWS:.1f}%",   "Remainder — no action this cycle"],
]
show_table("Playbook actions and population sizes", action_rows,
           ["Action", "Rows", "Slice share", "Operational meaning"])

display(Markdown("### 6.2 Archetype sizes (REFRESH tiering = VERY_STALE_HIGH_VIS first)"))
arch = ps["archetypes"]
arch_rows = [
    ["VERY_STALE_HIGH_VIS",   arch["VERY_STALE_HIGH_VIS"],   "Send REFRESH first — tiny, high-impact, high confidence stale × high-visibility overlap"],
    ["STRIKING_DISTANCE",     arch["STRIKING_DISTANCE"],     "Send REFRESH second — on the cusp of a visible rank change; human review is cheap and high-leverage"],
    ["CONTENT_DEPTH_GAP",     arch["CONTENT_DEPTH_GAP"],     "Send REFRESH third — candidate for a content-depth rework or section-add"],
    ["LOW_CTR_DECAY",         arch["LOW_CTR_DECAY"],         "Largest bucket; review in batches after the three small-cell archetypes above"],
    ["MIXED_SIGNALS",         arch["MIXED_SIGNALS"],         "Second largest; requires per-row triage after high-tier buckets are drained"],
]
show_table("Playbook archetypes (row-level segmentation used to rank the 500 REFRESH rows)",
           arch_rows, ["Archetype", "Rows (n)", "Tiering rule inside the REFRESH 500-cut"])

display(Markdown("### 6.3 Explicit no-go list (what the score must NOT be used for)"))
for item in ps["not_automated_list"]:
    print(f"  ✗  {item}")


## 7. Artifacts the paper embeds

Every figure and table the deployed page renders is generated by or copied from the weekly notebooks and committed under `work/outputs/` (CSV) or `docs/figures/` (PNG). The paper links to them directly from its `<figure>` and `<table>` elements.

### Figures
| Figure file (in `docs/figures/`) | What it shows |
|---|---|
| `paper_fig1_model_vs_baseline.png` | Frozen Week-4 hand-written rule vs. LogReg vs. Random Forest on the 5-fold honest GroupShuffleSplit. Bars = mean precision@200 / precision@50; error bars = ± 1 std. |
| `paper_fig2_split_inflation.png` | Same Random Forest on two split regimes. Left pair = Random ShuffleSplit (inflated); right pair = honest GroupShuffleSplit by client_id. Visualises the 14.2 pp split-inflation gap. |
| `paper_fig3_staleness_signal.png` | Severe-decline rate (bars, left y-axis) rising monotonically across three staleness buckets (<90d → 90–179d → ≥180d). Line with markers = median trend_pct (right axis). Confirms staleness directionality. |
| `paper_fig4_playbook_archetypes.png` | Five archetype sizes and the 500 / 1,500 / 20,006 REFRESH/OBSERVE/IGNORE stack relative to the slice. |

### Tables / CSVs
| Output file (in `work/outputs/`) | Purpose |
|---|---|
| `baseline_action_score.csv` | Frozen baseline score rows (Week-4 rule). Every row: action, score, reason-code, archetype. |
| `playbook_ranked_actions.csv` | Final playbook ranked rows (model score + action cut + archetype + confidence band label HIGH/MEDIUM/LOW). |

### JSON receipts referenced by the paper's claim set
Every numeric claim in §2–§6 can be reproduced from:
- `model_comparison.json` → §4 main table
- `validation_audit.json` → §3.5/§3.6 split regimes + leakage audit
- `baseline_metrics.json` → §3.3 baseline rule + §4.3 staleness signal
- `signal_audit_receipt.json` → §2 slice counts + §5 limitations
- `playbook_summary.json` → §6 action distribution + archetypes


In [ ]:
# --- 7. Artifact verification list (stdlib-only; no Pillow/PIL required) ---
import struct, os

DOCS = ROOT / "docs"
FIGS = DOCS / "figures"
OUT  = ROOT / "work" / "outputs"

def _read_image_dims(path):
    """Return (width, height) for PNG or JPEG files using stdlib struct only.

    Deliberately avoids the optional Pillow/PIL package, which may not be installed
    in stripped-down notebook environments. The paper figures are PNGs; this helper
    also handles JPEGs in case future figures use that format.

    Supported formats:

      PNG: reads the IHDR chunk. After the 8-byte PNG signature comes a 4-byte chunk
           length (BE uint32), then a 4-byte chunk type ("IHDR"), then 4 bytes width
           (BE uint32) and 4 bytes height (BE uint32) starting at absolute file offset 16.

      JPEG: scans SOF markers (0xC0..0xC3, 0xC5..0xC7, 0xC9..0xCB, 0xCD..0xCF). Each
            SOF segment stores precision(1) + height(BE uint16) + width(BE uint16) at
            offset 3 inside the segment payload.
    """
    path_str = path if isinstance(path, (str, bytes)) else str(path)
    try:
        with open(path_str, "rb") as f:
            head = f.read(24)
            if len(head) < 16:
                return ("?", "?")
            # --- PNG signature: 89 50 4E 47 0D 0A 1A 0A ---
            if (head[0] == 0x89
                    and head[1] == 0x50 and head[2] == 0x4E and head[3] == 0x47
                    and head[4] == 0x0D and head[5] == 0x0A
                    and head[6] == 0x1A and head[7] == 0x0A):
                f.seek(16)
                wh = f.read(8)
                if len(wh) == 8:
                    w, h = struct.unpack(">II", wh)
                    return (w, h)
                return ("?", "?")
            # --- JPEG SOI marker: FF D8 ---
            if head[0] == 0xFF and head[1] == 0xD8:
                f.seek(2)
                while True:
                    b = f.read(1)
                    if not b:
                        return ("?", "?")
                    if b[0] != 0xFF:
                        continue
                    # skip FF fill padding bytes
                    while True:
                        mb = f.read(1)
                        if not mb:
                            return ("?", "?")
                        if mb[0] != 0xFF:
                            break
                    marker = mb[0]
                    # standalone markers: SOI(0xD8), EOI(0xD9), RSTn(0xD0..0xD7), TEM(0x01)
                    if (marker == 0xD8 or marker == 0xD9 or marker == 0x01
                            or (0xD0 <= marker <= 0xD7)):
                        continue
                    len_bytes = f.read(2)
                    if len(len_bytes) != 2:
                        return ("?", "?")
                    seg_len = struct.unpack(">H", len_bytes)[0]
                    if (0xC0 <= marker <= 0xC3
                            or 0xC5 <= marker <= 0xC7
                            or 0xC9 <= marker <= 0xCB
                            or 0xCD <= marker <= 0xCF):
                        # precision(1) + height(BE uint16) + width(BE uint16)
                        payload = f.read(5)
                        if len(payload) == 5:
                            h, w = struct.unpack(">HH", payload[1:5])
                            return (w, h)
                        return ("?", "?")
                    # skip rest of segment (length includes the 2 length bytes)
                    remaining = seg_len - 2
                    if remaining > 0:
                        f.seek(remaining, 1)
                return ("?", "?")
            return ("?", "?")
    except Exception:
        return ("?", "?")

display(Markdown("### Figure files the paper embeds (docs/figures/*.png)"))
fig_files = [
    ("paper_fig1_model_vs_baseline.png",    "Frozen rule vs LogReg vs RF (5-fold honest GroupShuffleSplit, prec@200/prec@50,± 1 std)"),
    ("paper_fig2_split_inflation.png",      "Same RF on two split regimes (Random ShuffleSplit vs GroupShuffleSplit by client_id)"),
    ("paper_fig3_staleness_signal.png",    "Severe-decline rate + median trend_pct across three staleness buckets; monotonic confirmation"),
    ("paper_fig4_playbook_archetypes.png", "REFRESH / OBSERVE / IGNORE action stacks + 5 archetype sizes"),
]
fig_rows = []
for name, purpose in fig_files:
    path = FIGS / name
    if path.exists():
        w, h = _read_image_dims(path)
        size_kb = path.stat().st_size // 1024
        if isinstance(w, int) and isinstance(h, int):
            dims = f"{w}×{h}px"
        else:
            dims = "unknown"
        fig_rows.append([name, "✓ EXISTS", str(size_kb) + " KB", dims, purpose])
    else:
        fig_rows.append([name, "✗ MISSING", "-", "-", purpose])
show_table("Figures embedded in docs/index.html",
           fig_rows, ["File", "Status", "Size", "Dimensions", "What the figure shows"])

display(Markdown("### Artifact JSONs + CSVs referenced in cells 1 through 12 above"))
art_files = [
    ("work/outputs/model_comparison.json",   "Honest 5-fold model vs baseline comparison (prec@200, prec@50)"),
    ("work/outputs/baseline_metrics.json",   "Baseline rule components + staleness signal buckets"),
    ("work/outputs/validation_audit.json",   "Two split regimes + leakage harness confession + 9-point audit"),
    ("work/outputs/signal_audit_receipt.json","Slice setup counts + staleness/pos-times-SV verdicts + heavy-tail shares"),
    ("work/outputs/playbook_summary.json",   "Playbook actions + archetypes + HIGH/MEDIUM band cutoffs + no-go list"),
    ("work/outputs/baseline_action_score.csv","Frozen baseline rank output (Week 4)"),
    ("work/outputs/playbook_ranked_actions.csv","Final playbook rank output (model score + action + archetype)"),
]
art_rows = []
for rel, purpose in art_files:
    path = ROOT / rel
    art_rows.append([rel, "✓ EXISTS" if path.exists() else "✗ MISSING",
                     f"{path.stat().st_size // 1024} KB" if path.exists() else "—", purpose])
show_table("Weekly notebook outputs used by this capstone notebook",
           art_rows, ["File", "Exists", "Size", "Claim-area covered"])

# Count total references (evidence of completeness)
total_artifacts = sum(1 for _,_,x,_ in art_rows if "✓" in x) + sum(1 for _,x,_,_,_ in fig_rows if "✓" in x)
print()
print("Referenced artifacts present on disk:", total_artifacts, "of", str(len(fig_files) + len(art_files)))
print("Figure images total:", str(sum(1 for _,x,_,_,_ in fig_rows if "✓" in x)) + "/" + str(len(fig_files)))
print("CSV/JSON receipts total:", str(sum(1 for _,x,_,_ in art_rows if "✓" in x)) + "/" + str(len(art_files)))


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

---

## ML-12. Outreach artefacts (closing addendum; copy-paste as needed)

### 5-minute demo outline (for live walkthrough)
1. **Setup (0:00–0:30) — title, question, decision.** "We're trying to tell editors which 200 of 22,000 pages to review first this month." Live page URL + paper H1/subtitle.
2. **Data (0:30–1:15) — filters + timeline.** 30k starter rows → 22k after three simple rules. The 3-filter rationale and the Past → Decision → Future timeline diagram.
3. **Methodology (1:15–2:15) — seven questions, one minute on #5 validation and #6 leakage.** Emphasise client-held-out split and the +32.5 pp confession jump: "this is what honest validation protects against."
4. **Result (2:15–3:15) — main table + split-design figure.** 151 vs 106 severe-decline pages found at top-200. Then show: 89.8% looks great on a naive shuffle — 75.6% is the honest number.
5. **Playbook (3:15–4:00) — 500 REFRESH / 1,500 OBSERVE / 20,006 IGNORE.** Archetype tiering: VERY_STALE_HIGH_VIS first, then STRIKING_DISTANCE, then CONTENT_DEPTH_GAP.
6. **Limitations (4:00–4:30) — top 3 caveats in 30 seconds each.** Starter-export scope only, no forward-calendar holdout, probabilities ranked-not-calibrated.
7. **Close (4:30–5:00) — recommendations + artifacts.** "Top-500 first-by-archetype, retire the integer ranking, add a content-type flag, no automation without forward-calendar holdout pass."

### Social-post cut (LinkedIn / X copy — paste as needed)
> "Handed 22,006 content pages from 30 clients — and editors only have time for the top 200.
>
> I built a ranked-queue system for the FlyRank ML Internship capstone that compares a frozen hand-written baseline rule, a Logistic Regression, and a Random Forest on **client-held-out validation** (not just a random page shuffle). Honest numbers: RF precision@200 = 75.6% ± 8.4%, versus 52.9% ± 15.5% for the baseline — about 45 extra genuinely-declining pages in every 200-page monthly review.
>
> The honest GroupShuffleSplit result (75.6%) is 14.2 percentage points worse than a naive Random ShuffleSplit would have you believe. That 14.2 pp gap is exactly why grouped validation matters.
>
> Decision-support only, no automation, every claim traced to a committed receipt JSON. Live paper: https://flyrank-ml.vercel.app/ · code: https://github.com/Lakes41/flyrank-ml #MachineLearning #SEO #ContentStrategy #AIAudit #Validation"

### 3-sentence employer-facing summary
> I delivered a client-held-out ranked-queue ML system for a 22,006-page FlyRank ML Internship slice, with full grouped validation against a frozen hand-written rule. The chosen Random Forest increases precision@200 by +22.7 percentage points over the hand-written baseline at equivalent cost (200-page editorial capacity), translating into approximately 45 additional severe-decline pages surfaced per month for an editor's limited review cycle. Every numeric claim is reproducible from five committed JSON receipts under `work/outputs/`, and the output is decision-support only with explicit guardrails against automated page actions.
